Q1

In [7]:
from collections import deque

hospital = {
    "WardA": {"status": "Critical", "adjacent": ["WardB", "WardC"]},
    "WardB": {"status": "Normal", "adjacent": ["WardA", "WardD"]},
    "WardC": {"status": "Critical", "adjacent": ["WardA", "WardD"]},
    "WardD": {"status": "Empty", "adjacent": ["WardB", "WardC"]}
}

#1
class UtilityBasedAgent:
    def __init__(self, start):
        self.current = start
        self.moves = 0
        self.critical_treated = 0

    def utility(self):
        return (10 * self.critical_treated) - self.moves
#3
    def act(self):
        if hospital[self.current]["status"] == "Critical":
            print(f"Treating patient at {self.current}")
            hospital[self.current]["status"] = "Treated"
            self.critical_treated += 1

        else:
            path = bfs_nearest_critical(self.current)

            if path and len(path) > 1:
                next_move = path[1]
                print(f"Moving from {self.current} → {next_move}")
                self.current = next_move
                self.moves += 1

    def all_treated(self):
        for ward in hospital:
            if hospital[ward]["status"] == "Critical":
                return False
        return True
#2
def bfs_nearest_critical(start):
    queue = deque([(start, [start])])
    visited = set()

    while queue:
        node, path = queue.popleft()

        if node in visited:
            continue
        visited.add(node)

        if hospital[node]["status"] == "Critical":
            return path

        for neighbor in hospital[node]["adjacent"]:
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))

    return None
#4
agent = UtilityBasedAgent(start="WardA")

step = 1
while not agent.all_treated():
    print(f"\nStep {step}:")
    print(f"Current Ward: {agent.current}")
    agent.act()
    print(f"Moves: {agent.moves}, Treated: {agent.critical_treated}, Utility: {agent.utility()}")
    step += 1

print("\nFinal Utility:", agent.utility())


Step 1:
Current Ward: WardA
Treating patient at WardA
Moves: 0, Treated: 1, Utility: 10

Step 2:
Current Ward: WardA
Moving from WardA → WardC
Moves: 1, Treated: 1, Utility: 9

Step 3:
Current Ward: WardC
Treating patient at WardC
Moves: 1, Treated: 2, Utility: 19

Final Utility: 19


Q2

In [10]:
import heapq

#1
grid = [
    ['S', '.', '.', 'W', '.'],
    ['.', '#', '.', '.', '.'],
    ['.', '#', '.', '#', '.'],
    ['.', '.', 'F', '.', '.'],
    ['.', '.', '.', '.', 'G']
]

ROWS = len(grid)
COLS = len(grid[0])

def get_cost(cell):
    if cell == '.':
        return 1
    elif cell == 'W':
        return 3
    elif cell == 'F':
        return 5
    elif cell == 'S' or cell == 'G':
        return 1
    else:
        return float('inf')

#2
def heuristic(r, c, goal_pos):
    return abs(goal_pos[0] - r) + abs(goal_pos[1] - c)

def astar():
    start_node = (0, 0)
    goal_node = (4, 4)

    pq = []
    heapq.heappush(pq, (0, start_node))

    came_from = {}
    g_cost = {start_node: 0}

    visited = set()

    while pq:
        f, current = heapq.heappop(pq)

        if current in visited:
            continue
        visited.add(current)

        if current == goal_node:
            break

        r, c = current

        directions = [(1,0), (-1,0), (0,1), (0,-1)]

        for dr, dc in directions:
            nr, nc = r + dr, c + dc

            if 0 <= nr < ROWS and 0 <= nc < COLS:
                if grid[nr][nc] == '#':
                    continue

                new_g = g_cost[current] + get_cost(grid[nr][nc])

                if (nr, nc) not in g_cost or new_g < g_cost[(nr, nc)]:
                    g_cost[(nr, nc)] = new_g
                    f_cost = new_g + heuristic(nr, nc, goal_node)

                    heapq.heappush(pq, (f_cost, (nr, nc)))
                    came_from[(nr, nc)] = current
#3
    path = []
    curr = goal_node
    while curr != start_node:
        path.append(curr)
        curr = came_from[curr]
    path.append(start_node)
    path.reverse()

    return path, g_cost[goal_node]

path, cost = astar()

print("Optimal Path:")
for p in path:
    print(p)

print("\nTotal Cost:", cost)

Optimal Path:
(0, 0)
(0, 1)
(0, 2)
(1, 2)
(1, 3)
(1, 4)
(2, 4)
(3, 4)
(4, 4)

Total Cost: 8


Q3

In [13]:
import random
import math

cities = {
    0: (2, 3),
    1: (5, 7),
    2: (1, 9),
    3: (8, 2),
    4: (4, 5)
}

def calculate_distance(city1, city2):
    return math.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)

#1
def total_route_distance(route):
    dist = 0
    for i in range(len(route) - 1):
        dist += calculate_distance(cities[route[i]], cities[route[i+1]])
    dist += calculate_distance(cities[route[-1]], cities[route[0]])
    return dist

#2
def tournament_selection(population, k=3):
    selection = random.sample(population, k)
    selection.sort(key=lambda x: total_route_distance(x))
    return selection[0]

#3
def order_crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))

    child = [None] * size
    child[start:end+1] = parent1[start:end+1]

    p2_pointer = 0
    for i in range(size):
        if child[i] is None:
            while parent2[p2_pointer] in child:
                p2_pointer += 1
            child[i] = parent2[p2_pointer]
    return child

#4
def swap_mutation(route, mutation_rate=0.1):
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(route)), 2)
        route[idx1], route[idx2] = route[idx2], route[idx1]
    return route

#5
def run_genetic_algorithm(generations=100, pop_size=20):
    city_ids = list(cities.keys())
    population = [random.sample(city_ids, len(city_ids)) for _ in range(pop_size)]

    for gen in range(generations):
        new_population = []
        for _ in range(pop_size):
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)
            child = order_crossover(p1, p2)
            child = swap_mutation(child)
            new_population.append(child)

        population = new_population

        # Optional: Check if desired minimal distance is found
        best_current = min(population, key=total_route_distance)
        if total_route_distance(best_current) < 23.9: # Example threshold
            break

    best_route = min(population, key=total_route_distance)
    return best_route, total_route_distance(best_route)

best_route, best_dist = run_genetic_algorithm()
print(f"Best Route: {best_route}")
print(f"Total Distance: {best_dist:.2f} units")

Best Route: [1, 2, 0, 3, 4]
Total Distance: 23.87 units
